# SF Crime Category Prediction

**Goal:** Given a San Francisco Police Department (SFPD) incident record — timestamp, location, and police district — predict the crime category across 39 classes.

**Dataset:** [Kaggle SF Crime Classification](https://www.kaggle.com/competitions/sf-crime) — 878K labeled incidents from 2003–2015.

**Primary metric:** Multiclass log loss (rewards calibrated probability estimates, not just correct labels).

---
### Notebook outline
1. Data overview and quality checks  
2. Exploratory data analysis  
3. Feature engineering  
4. Model training and comparison  
5. Feature importance and error analysis  
6. Key findings  

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from xgboost import XGBClassifier

# Make src/ importable from the notebooks/ directory
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src import features as F
from src import evaluate as E
from src import visualize as V

FIGURES_DIR = os.path.join('..', 'outputs', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('Environment ready.')

## 1. Data Overview

In [ ]:
df_raw = pd.read_csv(os.path.join('..', 'data', 'train.csv'))

print(f'Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'Columns: {df_raw.columns.tolist()}')
print(f'Date range: {df_raw["Dates"].min()} → {df_raw["Dates"].max()}')
print(f'Crime categories: {df_raw["Category"].nunique()}')
df_raw.head(3)

In [ ]:
# Missing values and coordinate anomalies
print('Missing values per column:')
print(df_raw.isnull().sum())
print()
print('Coordinate statistics (note the Y=90 outliers):')
print(df_raw[['X', 'Y']].describe())

In [ ]:
# Remove 67 records where Y=90 (a sentinel for missing GPS coordinates)
df = F.clean_coordinates(df_raw)
print(f'After coordinate cleaning: {len(df):,} records ({len(df_raw) - len(df):,} removed)')

## 2. Exploratory Data Analysis

In [ ]:
V.plot_crime_distribution(df, top_n=15)

The distribution is heavily skewed: `LARCENY/THEFT` alone accounts for ~21% of incidents. This class imbalance means a naive majority-class classifier would score ~21% accuracy — establishing a meaningful floor.

39 categories span a wide behavioral spectrum, from rare offenses (TREA, PORNOGRAPHY: <50 records each) to thousands of daily incidents. Any model must handle this imbalance without collapsing rare classes.

In [ ]:
# Parse dates before temporal EDA
df = F.extract_temporal_features(df)
V.plot_temporal_patterns(df)

**Observations:**
- Crime peaks at **17:00–18:00** (end of workday) and has a secondary peak around noon. The 5–6 AM trough is the quietest window.
- **Friday and Saturday** drive significantly more incidents than weekdays — a clear signal for the IsWeekend feature.
- Monthly volume is stable; January shows a slight dip consistent with holiday seasonality.

In [ ]:
V.plot_district_heatmap(df, top_categories=10)

Different districts have markedly different crime mixes — the **TENDERLOIN** district is disproportionately DRUG/NARCOTIC and PROSTITUTION; **RICHMOND** and **PARK** see more VEHICLE THEFT as a share. Police district is therefore a strong categorical predictor.

## 3. Feature Engineering

Two feature groups are constructed:

| Group | Features | Rationale |
|-------|----------|-----------|
| **Temporal** | `Hour`, `DayOfWeek_Num`, `IsWeekend`, `Month`, `HourBlock`, `Season` | Crime type varies strongly by time of day and week |
| **Spatial** | `DistanceFromDowntown_km`, `GeoCluster` (K=15 KMeans) | Neighborhood context — downtown vs residential vs industrial |
| **Categorical** | `PdDistrict`, `DayOfWeek` | District encodes policing patterns; raw day name for OHE |

> **Note:** `Descript` and `Resolution` are deliberately excluded. They contain information that would not be available at dispatch time (and would constitute label leakage for `Resolution`).

In [ ]:
# Fit KMeans on all data, then derive spatial features
df, geo_kmeans = F.add_spatial_features(df)
print(f'Feature columns: {F.NUMERIC_FEATURES + F.CATEGORICAL_FEATURES}')
F.build_feature_matrix(df).head(3)

## 4. Model Training and Evaluation

### Train / validation split

A **temporal split** is used: odd calendar weeks form the training set, even weeks form validation. This prevents any data leakage from future incidents into training and is more realistic than a random split for time-series incident data.

In [ ]:
from scripts.train import temporal_split, build_preprocessor, build_pipelines

train_df, val_df = temporal_split(df)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')

le = LabelEncoder()
le.fit(df['Category'])
y_train_full = le.transform(train_df['Category'])
y_val_full   = le.transform(val_df['Category'])

# Subsample for interactive use
rng = np.random.default_rng(42)
TRAIN_N, VAL_N = 50_000, 20_000
ti = rng.choice(len(train_df), TRAIN_N, replace=False)
vi = rng.choice(len(val_df),   VAL_N,   replace=False)

X_train = F.build_feature_matrix(train_df).iloc[ti]
y_train = y_train_full[ti]
X_val   = F.build_feature_matrix(val_df).iloc[vi]
y_val   = y_val_full[vi]

# Patch rare classes missing from sample
missing = np.setdiff1d(np.arange(len(le.classes_)), np.unique(y_train))
if len(missing):
    extra_idx = [np.where(y_train_full == c)[0][0] for c in missing]
    X_train = pd.concat([X_train, F.build_feature_matrix(train_df).iloc[extra_idx]])
    y_train = np.concatenate([y_train, missing])
print(f'Training sample: {len(X_train):,} | Validation sample: {len(X_val):,}')

In [ ]:
preprocessor = build_preprocessor()
pipelines = build_pipelines(preprocessor)

In [ ]:
# --- Logistic Regression ---
lr_pipe = pipelines['Logistic Regression']
lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_val)
res_lr = E.evaluate(y_val, y_pred_lr, lr_pipe.predict_proba(X_val), len(le.classes_), list(le.classes_))
print(f"Logistic Regression — Accuracy: {res_lr['accuracy']:.1%} | Log Loss: {res_lr['log_loss']:.4f}")

In [ ]:
# --- Random Forest ---
rf_pipe = pipelines['Random Forest']
rf_pipe.fit(X_train, y_train)
y_pred_rf = rf_pipe.predict(X_val)
res_rf = E.evaluate(y_val, y_pred_rf, rf_pipe.predict_proba(X_val), len(le.classes_), list(le.classes_))
print(f"Random Forest — Accuracy: {res_rf['accuracy']:.1%} | Log Loss: {res_rf['log_loss']:.4f}")

In [ ]:
# --- XGBoost (requires remapped labels for classes present in sample) ---
xgb_pipe = pipelines['XGBoost']
xgb_le = LabelEncoder()
xgb_le.fit(np.unique(y_train))
xgb_pipe.fit(X_train, xgb_le.transform(y_train))

y_pred_xgb = xgb_le.inverse_transform(xgb_pipe.predict(X_val))
xgb_proba_raw = xgb_pipe.predict_proba(X_val)
xgb_proba_full = np.full((len(y_val), len(le.classes_)), 1e-9)
xgb_proba_full[:, xgb_le.classes_] = xgb_proba_raw
xgb_proba_full /= xgb_proba_full.sum(axis=1, keepdims=True)

res_xgb = E.evaluate(y_val, y_pred_xgb, xgb_proba_full, len(le.classes_), list(le.classes_))
print(f"XGBoost — Accuracy: {res_xgb['accuracy']:.1%} | Log Loss: {res_xgb['log_loss']:.4f}")

In [ ]:
# --- KNN ---
knn_pipe = pipelines['KNN']
knn_pipe.fit(X_train, y_train)
y_pred_knn = knn_pipe.predict(X_val)
res_knn = E.evaluate(y_val, y_pred_knn, knn_pipe.predict_proba(X_val), len(le.classes_), list(le.classes_))
print(f"KNN — Accuracy: {res_knn['accuracy']:.1%} | Log Loss: {res_knn['log_loss']:.4f}")

In [ ]:
# --- Naive Bayes ---
nb_pipe = pipelines['Naive Bayes']
nb_pipe.fit(X_train, y_train)
y_pred_nb = nb_pipe.predict(X_val)
res_nb = E.evaluate(y_val, y_pred_nb, nb_pipe.predict_proba(X_val), len(le.classes_), list(le.classes_))
print(f"Naive Bayes — Accuracy: {res_nb['accuracy']:.1%} | Log Loss: {res_nb['log_loss']:.4f}")

## 5. Results

In [ ]:
all_results = {
    'Logistic Regression': res_lr,
    'Random Forest':       res_rf,
    'XGBoost':             res_xgb,
    'KNN':                 res_knn,
    'Naive Bayes':         res_nb,
}
summary = E.summarise_results(all_results)
print('Model Comparison (sorted by log loss):')
summary

In [ ]:
V.plot_model_comparison(summary)

**XGBoost achieves the best log loss (2.53) and accuracy (29%).** Logistic regression is competitive on log loss (2.59) despite being far simpler. Random Forest and KNN underperform on log loss despite reasonable accuracy, suggesting their probability estimates are poorly calibrated for rare classes.

In [ ]:
# Feature importance from XGBoost
ohe = xgb_pipe.named_steps['pre'].named_transformers_['cat']
cat_names = list(ohe.get_feature_names_out(F.CATEGORICAL_FEATURES))
feat_names = F.NUMERIC_FEATURES + cat_names
imp_df = pd.DataFrame({'Feature': feat_names, 'Importance': xgb_pipe.named_steps['clf'].feature_importances_})
imp_df = imp_df.sort_values('Importance', ascending=False).reset_index(drop=True)
V.plot_feature_importance(imp_df, top_n=15)

**`DistanceFromDowntown_km` dominates** with ~63% of total importance, followed by season one-hot features and day-of-week. This suggests spatial proximity to the city center is the single strongest structural predictor of crime type — downtown crimes skew heavily toward theft and fraud, while outlying areas see more vehicle theft and residential burglary.

In [ ]:
# Confusion matrix for top-8 categories (XGBoost)
sorted_cats = df['Category'].value_counts().index.tolist()
top8_idx = [le.transform([c])[0] for c in sorted_cats[:8]]

cm_full = confusion_matrix(y_val, y_pred_xgb)
cm_top = cm_full[np.ix_(top8_idx, top8_idx)]

fig, ax = plt.subplots(figsize=(10, 8))
labels_short = [c[:18] for c in sorted_cats[:8]]
sns.heatmap(cm_top, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_short, yticklabels=labels_short, ax=ax)
ax.set_title('Confusion Matrix — Top 8 Categories (XGBoost)', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.tick_params(axis='x', rotation=40)
plt.tight_layout()
plt.show()

## 6. Key Findings

| Finding | Detail |
|---------|--------|
| **Best model** | XGBoost: 29.0% accuracy, 2.53 log loss |
| **Most important feature** | `DistanceFromDowntown_km` — 63% of XGBoost importance |
| **Hardest classes to predict** | Rare categories (TREA, GAMBLING, BAD CHECKS) — almost always misclassified |
| **Easiest to separate** | DRUG/NARCOTIC and LARCENY/THEFT — highest per-class F1 |
| **Why accuracy is low** | 39 imbalanced classes with overlapping spatio-temporal signals; prediction difficulty is inherent to the domain |

### What would improve performance

- **Address-level features:** Street intersections carry strong prior information (e.g., 6th & Market vs Sunset Blvd).
- **Temporal trends:** Year-over-year trend features could capture gentrification effects shifting crime patterns.
- **Richer spatial encoding:** Replace KMeans clusters with census tract or neighborhood boundary joins.
- **Calibration:** Post-hoc probability calibration (Platt scaling / Isotonic) would reduce log loss without changing predictions.